
## 1. Load data

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # add project root to path

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data.data_ingestion  import DataIngestion
from src.data.data_preprocessing  import DataPreprocessor
from src.features.build_features   import FeatureEngineer
#from src.features.feature_selection import FeatureSelector
from src.visualization.eda_plots   import EDAVisualizer
from src.utils.config             import load_config

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)


cfg    = load_config("../config.yaml")
TARGET = cfg["data"]["target_column"]          # "Class"
DROP   = cfg["data"].get("id_column", [])   
RS     = cfg["training"]["random_state"]        # 42
viz    = EDAVisualizer()

In [2]:
ingestor = DataIngestion(config_path="../config.yaml")
df = ingestor.load(path="../data/raw/panic_disorder_dataset.csv")


2026-10-02 16:30:25 | INFO     | src.data.data_ingestion | Loading data from: E:\Python projects\Healthcare_research\data\raw\panic_disorder_dataset.csv
2026-10-02 16:30:25 | INFO     | src.data.data_ingestion | Loaded  shape  : (120000, 17)
2026-10-02 16:30:26 | INFO     | src.data.data_ingestion | Memory usage   : 107.67 MB
2026-10-02 16:30:26 | INFO     | src.data.data_ingestion | Target 'Panic Disorder Diagnosis' — Mean: 0.04



## 2. Split FIRST — Before Any Fitting


In [3]:
test_size  = cfg["split"]["test_size"]   # 0.15
val_size   = cfg["split"]["val_size"]    # 0.15
stratify   = cfg["split"].get("stratify", True)

if isinstance(DROP, str):
    DROP = [DROP]
    
X_raw = df.drop(columns=[TARGET] + DROP, errors="ignore")
y_raw = df[TARGET]

# ── First cut: carve off test (15%) ──────────────────────────────
X_temp, X_test_raw, y_temp, y_test = train_test_split(
    X_raw, y_raw,
    test_size=test_size,
    random_state=RS,
    stratify=y_raw if stratify else None
)

# ── Second cut: carve val from remaining 85% ─────────────────────
# We want val = 15% of total
# val_ratio = 0.15 / (1.0 - 0.15) = 0.1765 of the remaining 85%
val_ratio = val_size / (1.0 - test_size)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_ratio,
    random_state=RS,
    stratify=y_temp if stratify else None
)

print("=== Split summary (BEFORE any fitting) ===")
print(f"  Train : {X_train_raw.shape}  | fraud: {y_train.sum():,} ({y_train.mean():.4%})")
print(f"  Val   : {X_val_raw.shape}  | fraud: {y_val.sum():,}   ({y_val.mean():.4%})")
print(f"  Test  : {X_test_raw.shape}  | fraud: {y_test.sum():,}   ({y_test.mean():.4%})")
print()
print("Verify: all three fraud rates should be ~0.173% (stratification check)")

=== Split summary (BEFORE any fitting) ===
  Train : (84000, 15)  | fraud: 3,588 (4.2714%)
  Val   : (18000, 15)  | fraud: 769   (4.2722%)
  Test  : (18000, 15)  | fraud: 769   (4.2722%)

Verify: all three fraud rates should be ~0.173% (stratification check)


---
## 3. Data Preprocessing — Fit on Train Only


In [4]:
preprocessor = DataPreprocessor(config_path="../config.yaml")

# Build DataFrames for preprocessor (needs target column present)
train_raw_df = X_train_raw.copy()
train_raw_df[TARGET] = y_train.values

val_raw_df = X_val_raw.copy()
val_raw_df[TARGET] = y_val.values

test_raw_df = X_test_raw.copy()
test_raw_df[TARGET] = y_test.values

train_clean = preprocessor.fit_transform(train_raw_df)
val_clean   = preprocessor.transform(val_raw_df)
test_clean  = preprocessor.transform(test_raw_df)

print("=== Cleaning report (train set) ===")
for k, v in preprocessor.cleaning_report.items():
    print(f"  {k:<25} : {v}")

print(f"\nTrain shape before : {X_train_raw.shape}")
print(f"Train shape after  : {train_clean.shape}")

print(f"\nValidation shape before : {X_val_raw.shape}")
print(f"Validation shape after  : {val_clean.shape}")

print(f"\nTest shape before : {X_test_raw.shape}")
print(f"Test shape after  : {test_clean.shape}")

print(f"\ny_train shape : {y_train.shape}")
print(f"y_val shape   : {y_val.shape}")
print(f"y_test shape  : {y_test.shape}")


2026-10-02 16:30:32 | INFO     | src.data.data_preprocessing | === Data Preprocessing START ===


2026-10-02 16:30:32 | INFO     | src.data.data_preprocessing | Handling 70,120 missing values (numerical: median, categorical: mode)
2026-10-02 16:30:32 | INFO     | src.data.data_preprocessing | Preprocessing complete. 84,000 → 84,000 rows (0 removed)
2026-10-02 16:30:32 | INFO     | src.data.data_preprocessing | Handling 14,993 missing values (numerical: median, categorical: mode)
2026-10-02 16:30:32 | INFO     | src.data.data_preprocessing | Handling 14,962 missing values (numerical: median, categorical: mode)


=== Cleaning report (train set) ===
  original_shape            : (84000, 16)
  clean_shape               : (84000, 16)
  rows_removed              : 0

Train shape before : (84000, 15)
Train shape after  : (84000, 16)

Validation shape before : (18000, 15)
Validation shape after  : (18000, 16)

Test shape before : (18000, 15)
Test shape after  : (18000, 16)

y_train shape : (84000,)
y_val shape   : (18000,)
y_test shape  : (18000,)



## 4. Feature Engineering — Fit on Train Only


In [9]:
# Separate features from target — feature engineer never sees the target
X_train_clean = train_clean.drop(columns=[TARGET], errors="ignore")
X_val_clean   = val_clean.drop(columns=[TARGET], errors="ignore")
X_test_clean  = test_clean.drop(columns=[TARGET], errors="ignore")

fe = FeatureEngineer(config_path="../config.yaml")

# FIT on training features only
X_train_eng = fe.fit_transform(X_train_clean)

# APPLY to val and test — uses transformer learned from train
X_val_eng = fe.transform(X_val_clean)
X_test_eng  = fe.transform(X_test_clean)

print(f"Features before engineering : {X_train_clean.shape[1]}")
print(f"Features after  engineering : {X_train_eng.shape[1]}")

print("\nValidation features after engineering :", X_val_eng.shape[1])
print("Test features after engineering       :", X_test_eng.shape[1])

    
original_columns = set(X_train_clean.columns)
final_columns = set(X_train_eng.columns)

new_columns = sorted(final_columns - original_columns)

print("\nNew columns after encoding:")
for col in new_columns:
    print(f"  {col}")



2026-10-02 16:38:36 | INFO     | src.features.build_features | === Feature Engineering FIT ===
2026-10-02 16:38:36 | INFO     | src.features.build_features | Categorical encoding applied to 14 column(s).
2026-10-02 16:38:36 | INFO     | src.features.build_features | Feature engineering fitted. Output features: 27
2026-10-02 16:38:36 | INFO     | src.features.build_features | === Feature Engineering TRANSFORM ===
2026-10-02 16:38:37 | INFO     | src.features.build_features | Categorical encoding applied to 14 column(s).
2026-10-02 16:38:37 | INFO     | src.features.build_features | === Feature Engineering TRANSFORM ===
2026-10-02 16:38:37 | INFO     | src.features.build_features | Categorical encoding applied to 14 column(s).
2026-10-02 16:38:37 | INFO     | src.features.build_features | === Feature Engineering TRANSFORM ===
2026-10-02 16:38:37 | INFO     | src.features.build_features | Categorical encoding applied to 14 column(s).


Features before engineering : 15
Features after  engineering : 27

Validation features after engineering : 27
Test features after engineering       : 27

New columns after encoding:
  Coping Mechanisms_Meditation
  Coping Mechanisms_Seeking therapy
  Coping Mechanisms_Socializing
  Current Stressors_Low
  Current Stressors_Moderate
  Impact on Life_Moderate
  Impact on Life_Significant
  Lifestyle Factors_Exercise
  Lifestyle Factors_Sleep quality
  Medical History_Diabetes
  Medical History_Heart disease
  Psychiatric History_Bipolar disorder
  Psychiatric History_Depressive disorder
  Severity_Moderate
  Severity_Severe
  Social Support_Low
  Social Support_Moderate
  Symptoms_Dizziness
  Symptoms_Fear of losing control
  Symptoms_Panic attacks
  Symptoms_Shortness of breath


In [10]:
# Save processed datasets
fe.save_parquets(
    X_train=X_train_eng,
    X_val=X_val_eng,
    X_test=X_test_eng,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)

2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved X_train.parquet | shape=(84000, 27)
2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved X_val.parquet | shape=(18000, 27)
2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved X_test.parquet | shape=(18000, 27)
2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved y_train.parquet | shape=(84000, 1)
2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved y_val.parquet | shape=(18000, 1)
2026-10-02 16:39:05 | INFO     | src.features.build_features | Saved y_test.parquet | shape=(18000, 1)


### 5.1 Spot-check engineered features (train set)

In [ ]:
print("=== New features — first 5 rows ===")
print(X_train_eng[new_columns].head())

=== New features — first 5 rows ===
        Coping Mechanisms_Meditation  Coping Mechanisms_Seeking therapy  \
45171                           True                              False   
38914                          False                              False   
42359                          False                               True   
49367                          False                               True   
110728                         False                              False   

        Coping Mechanisms_Socializing  Current Stressors_Low  \
45171                           False                  False   
38914                           False                  False   
42359                           False                  False   
49367                           False                   True   
110728                           True                  False   

        Current Stressors_Moderate  Impact on Life_Moderate  \
45171                        False                    False   
38